# End-to-end NanoSIMS NRRD pipeline — Python only

This notebook replaces **both external FIJI/ImageJ registration stages** in the supplied workflow and runs from raw OpenMIMS/NanoSIMS NRRD files to registered per-mass TIFF stacks without launching FIJI.

The pipeline:

1. Reads every `.nrrd` or `.nhdr` file in `input_nrrd/` and orders files by a `sectNN` number when present.
2. Interprets the OpenMIMS NRRD axes as **X, Y, acquisition plane, mass** and reads mass names from `Mims_mass_symbols`.
3. Aligns acquisition planes inside each NRRD on the selected registration mass using SIFT features, the macro's nearest-neighbor ratio, rigid RANSAC, consecutive frame-to-frame matching, cumulative transforms, and bilinear interpolation. Each estimated transform is applied identically to all masses.
4. Sums the aligned acquisition planes to one image per mass and section.
5. Builds one serial-section stack per mass.
6. Optionally normalizes each section to the mean intensity of its mass stack. This follows the **actual code** in the supplied notebook: `stack_mean / section_mean`.
7. Writes an ImageJ-readable unregistered hyperstack.
8. Aligns serial sections on `12C14N` by default, again applying the same rigid transform to every mass.
9. Writes the registered hyperstack, separate registered mass stacks, transform matrices, CSV quality-control tables, and before/after montage images.

### FIJI-to-Python replacement map

| Supplied FIJI/notebook operation | Python replacement here |
|---|---|
| Re-order Hyperstack | Direct NRRD axis parsing and conversion to `ZCYX` |
| Linear Stack Alignment with SIFT MultiChannel | OpenCV SIFT + ratio filtering + rigid RANSAC + one transform for all channels |
| Z Project: Sum Slices | `numpy.sum` after plane registration |
| Hyperstack to Stack / TIFF save | `tifffile.imwrite(..., imagej=True)` |
| Manual second FIJI alignment on channel 3 | The same multichannel registration function, run across serial sections on mass `12C14N` |
| Split registered 4-D hyperstack | Direct channel extraction and TIFF writing |

> **Numerical-equivalence note:** the registration model and execution order mirror the supplied macro, but OpenCV's SIFT implementation is not the same code as FIJI's JavaSIFT implementation. Therefore, transforms should be functionally comparable but are not expected to be bit-for-bit identical to FIJI output. Always inspect the generated QC tables and montages before quantitative analysis.

## Dependencies

The notebook uses NumPy, pandas, Matplotlib, tifffile, and OpenCV. It contains its own NRRD reader, so `pynrrd`, `pystackreg`, ImageJ, and FIJI are not required.

Uncomment and run the line below once if these packages are absent, then restart the kernel:

```python
%pip install -U numpy pandas matplotlib tifffile opencv-python-headless
```

In [ ]:
from __future__ import annotations

import bz2
import gzip
import json
import math
import os
import re
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Iterable, Sequence

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tifffile


@dataclass(frozen=True)
class SIFTParameters:
    initial_gaussian_blur: float = 1.60
    steps_per_scale_octave: int = 3
    minimum_image_size: int = 64
    maximum_image_size: int = 1024
    feature_descriptor_size: int = 4
    feature_descriptor_orientation_bins: int = 8
    closest_next_closest_ratio: float = 0.92
    maximal_alignment_error: float = 25.0
    minimum_inlier_ratio: float = 0.05
    max_trials: int = 1000
    max_feature_scale_ratio: float = 1.5
    contrast_threshold: float = 0.04
    edge_threshold: float = 10.0
    registration_percentiles: tuple[float, float] = (0.5, 99.5)
    random_seed: int = 1729


NRRD_DTYPES: dict[str, np.dtype] = {
    "signed char": np.dtype(np.int8),
    "int8": np.dtype(np.int8),
    "int8_t": np.dtype(np.int8),
    "uchar": np.dtype(np.uint8),
    "unsigned char": np.dtype(np.uint8),
    "uint8": np.dtype(np.uint8),
    "uint8_t": np.dtype(np.uint8),
    "short": np.dtype(np.int16),
    "short int": np.dtype(np.int16),
    "signed short": np.dtype(np.int16),
    "signed short int": np.dtype(np.int16),
    "int16": np.dtype(np.int16),
    "int16_t": np.dtype(np.int16),
    "ushort": np.dtype(np.uint16),
    "unsigned short": np.dtype(np.uint16),
    "unsigned short int": np.dtype(np.uint16),
    "uint16": np.dtype(np.uint16),
    "uint16_t": np.dtype(np.uint16),
    "int": np.dtype(np.int32),
    "signed int": np.dtype(np.int32),
    "int32": np.dtype(np.int32),
    "int32_t": np.dtype(np.int32),
    "uint": np.dtype(np.uint32),
    "unsigned int": np.dtype(np.uint32),
    "uint32": np.dtype(np.uint32),
    "uint32_t": np.dtype(np.uint32),
    "longlong": np.dtype(np.int64),
    "long long": np.dtype(np.int64),
    "int64": np.dtype(np.int64),
    "int64_t": np.dtype(np.int64),
    "ulonglong": np.dtype(np.uint64),
    "unsigned long long": np.dtype(np.uint64),
    "uint64": np.dtype(np.uint64),
    "uint64_t": np.dtype(np.uint64),
    "float": np.dtype(np.float32),
    "double": np.dtype(np.float64),
}


def _header_lookup(header: dict[str, str], key: str, default: str | None = None) -> str | None:
    key_fold = key.casefold()
    for candidate, value in header.items():
        if candidate.casefold() == key_fold:
            return value
    return default


def _parse_nrrd_header(header_lines: Sequence[str]) -> dict[str, str]:
    header: dict[str, str] = {}
    for line in header_lines:
        if not line or line.startswith("#"):
            continue
        if ":=" in line:
            key, value = line.split(":=", 1)
        elif ":" in line:
            key, value = line.split(":", 1)
        else:
            continue
        header[key.strip()] = value.strip()
    return header


def read_nrrd(path: str | Path) -> tuple[np.ndarray, dict[str, str]]:
    """Read an embedded or detached NRRD without requiring pynrrd.

    Axis 0 is reconstructed as the fastest-varying NRRD axis (Fortran order),
    as required by the NRRD specification. Raw, gzip, bzip2, ASCII, and hex
    encodings are supported.
    """
    path = Path(path)
    with path.open("rb") as stream:
        magic = stream.readline().decode("ascii", errors="strict").strip()
        if not magic.startswith("NRRD"):
            raise ValueError(f"{path} is not an NRRD file (magic={magic!r}).")

        header_lines: list[str] = []
        while True:
            line = stream.readline()
            if not line:
                raise ValueError(f"Unexpected end of file while reading NRRD header: {path}")
            if line in (b"\n", b"\r\n"):
                break
            header_lines.append(line.decode("utf-8", errors="replace").rstrip("\r\n"))
        embedded_payload = stream.read()

    header = _parse_nrrd_header(header_lines)
    type_name = (_header_lookup(header, "type") or "").casefold()
    if type_name not in NRRD_DTYPES:
        raise NotImplementedError(f"Unsupported NRRD type {type_name!r} in {path.name}.")
    dtype = NRRD_DTYPES[type_name]

    sizes_text = _header_lookup(header, "sizes")
    if not sizes_text:
        raise ValueError(f"NRRD 'sizes' field is missing in {path.name}.")
    sizes = tuple(int(token) for token in sizes_text.split())
    dimension = int(_header_lookup(header, "dimension", str(len(sizes))) or len(sizes))
    if dimension != len(sizes):
        raise ValueError(f"NRRD dimension={dimension}, but sizes has {len(sizes)} axes in {path.name}.")

    if dtype.itemsize > 1:
        endian = (_header_lookup(header, "endian", "little") or "little").casefold()
        if endian not in {"little", "big"}:
            raise ValueError(f"Unsupported endian value {endian!r} in {path.name}.")
        file_dtype = dtype.newbyteorder("<" if endian == "little" else ">")
    else:
        file_dtype = dtype

    data_file = _header_lookup(header, "data file")
    if data_file:
        if data_file.upper().startswith("LIST"):
            raise NotImplementedError("NRRD LIST-style detached data is not supported by this notebook.")
        payload = (path.parent / data_file.strip().strip('"')).read_bytes()
    else:
        payload = embedded_payload

    encoding = (_header_lookup(header, "encoding", "raw") or "raw").casefold()
    if encoding in {"gzip", "gz"}:
        payload = gzip.decompress(payload)
    elif encoding in {"bzip2", "bz2"}:
        payload = bz2.decompress(payload)

    line_skip = int(_header_lookup(header, "line skip", "0") or 0)
    if line_skip:
        lines = payload.splitlines(keepends=True)
        payload = b"".join(lines[line_skip:])

    expected_count = int(np.prod(sizes, dtype=np.int64))
    if encoding in {"ascii", "text", "txt"}:
        values = np.fromstring(payload.decode("ascii"), sep=" ", dtype=file_dtype)
    elif encoding == "hex":
        values = np.frombuffer(bytes.fromhex(payload.decode("ascii")), dtype=file_dtype)
    elif encoding == "raw":
        byte_skip = int(_header_lookup(header, "byte skip", "0") or 0)
        expected_bytes = expected_count * file_dtype.itemsize
        if byte_skip == -1:
            payload = payload[-expected_bytes:]
        elif byte_skip > 0:
            payload = payload[byte_skip:]
        values = np.frombuffer(payload, dtype=file_dtype, count=expected_count)
    else:
        raise NotImplementedError(f"Unsupported NRRD encoding {encoding!r} in {path.name}.")

    if values.size < expected_count:
        raise ValueError(
            f"NRRD payload is short in {path.name}: expected {expected_count} values, got {values.size}."
        )
    if values.size > expected_count:
        values = values[:expected_count]

    data = values.reshape(sizes, order="F")
    # Convert non-native endian arrays to native endian for OpenCV/NumPy operations.
    data = data.astype(dtype.newbyteorder("="), copy=False)
    return data, header


def mass_names_from_header(header: dict[str, str]) -> list[str]:
    symbols = _header_lookup(header, "Mims_mass_symbols")
    if not symbols:
        raise ValueError("Mims_mass_symbols is missing from the NRRD header.")
    names = symbols.split()
    if not names:
        raise ValueError("Mims_mass_symbols is empty.")
    return names


def load_openmims_nrrd(path: str | Path) -> tuple[np.ndarray, dict[str, str], list[str]]:
    """Return OpenMIMS data in (plane, mass/channel, y, x) order."""
    data, header = read_nrrd(path)
    mass_names = mass_names_from_header(header)
    if data.ndim != 4:
        raise ValueError(f"Expected a 4-D OpenMIMS NRRD, got shape {data.shape} in {Path(path).name}.")

    kinds = (_header_lookup(header, "kinds", "") or "").split()
    list_axes = [i for i, kind in enumerate(kinds) if kind.casefold() in {"list", "vector"}]
    size_axes = [i for i, size in enumerate(data.shape) if size == len(mass_names)]
    if len(list_axes) == 1:
        mass_axis = list_axes[0]
    elif len(size_axes) == 1:
        mass_axis = size_axes[0]
    elif data.shape[-1] == len(mass_names):
        mass_axis = data.ndim - 1
    else:
        raise ValueError(
            f"Cannot identify the mass axis in {Path(path).name}; shape={data.shape}, masses={len(mass_names)}, kinds={kinds}."
        )

    xyzc = np.moveaxis(data, mass_axis, -1)
    if xyzc.shape[-1] != len(mass_names):
        raise AssertionError("Mass-axis reordering failed.")
    # The remaining OpenMIMS axes are X, Y, acquisition plane. NumPy images use Y, X.
    zcyx = np.transpose(xyzc, (2, 3, 1, 0))
    return np.ascontiguousarray(zcyx), header, mass_names


_SECTION_RE = re.compile(r"(?i)(?:sect(?:ion)?)[-_ ]?(\d+)")
_NATURAL_RE = re.compile(r"(\d+)")


def section_sort_key(path: Path) -> tuple[Any, ...]:
    match = _SECTION_RE.search(path.stem)
    if match:
        return (0, int(match.group(1)), path.name.casefold())
    natural = tuple(int(token) if token.isdigit() else token.casefold() for token in _NATURAL_RE.split(path.name))
    return (1, *natural)


def section_number(path: Path) -> int | None:
    match = _SECTION_RE.search(path.stem)
    return int(match.group(1)) if match else None


def safe_name(text: str) -> str:
    return re.sub(r"[^0-9A-Za-z._+-]+", "_", text).strip("_") or "unnamed"


## Registration engine

The default values reproduce the parameters in `SIFT_align_planes_of_nrrd_to_hyperstack.ijm` where there is a direct OpenCV equivalent:

- initial Gaussian blur: `1.60`
- steps per octave: `3`
- maximum image size: `1024`
- descriptor layout: `4 × 4` cells and `8` orientation bins
- closest/next-closest ratio: `0.92`
- maximum RANSAC error: `25 px`
- minimum inlier ratio: `0.05`
- rigid transformation
- bilinear interpolation

The FIJI implementation also hard-codes 1000 RANSAC trials and a 1.5 feature-scale ratio; those values are retained. OpenCV does not expose FIJI's `minimum_image_size` octave control one-for-one, so that setting is recorded for provenance while OpenCV manages its lower pyramid limit internally.

In [ ]:
def prepare_registration_image(image: np.ndarray, percentiles: tuple[float, float]) -> np.ndarray:
    image = np.asarray(image, dtype=np.float32)
    finite = image[np.isfinite(image)]
    if finite.size == 0:
        return np.zeros(image.shape, dtype=np.uint8)
    low_p, high_p = percentiles
    if not (0 <= low_p < high_p <= 100):
        raise ValueError(f"Invalid registration percentiles: {percentiles}")
    low, high = np.percentile(finite, (low_p, high_p))
    if not np.isfinite(low) or not np.isfinite(high) or high <= low:
        low = float(np.min(finite))
        high = float(np.max(finite))
    if high <= low:
        return np.zeros(image.shape, dtype=np.uint8)
    scaled = np.clip((image - low) / (high - low), 0.0, 1.0)
    scaled[~np.isfinite(scaled)] = 0.0
    return np.rint(scaled * 255.0).astype(np.uint8)


@dataclass
class FeatureSet:
    keypoints_xy: np.ndarray
    sizes: np.ndarray
    descriptors: np.ndarray | None
    n_features: int


def build_sift(params: SIFTParameters) -> cv2.SIFT:
    if not hasattr(cv2, "SIFT_create"):
        raise RuntimeError(
            "This OpenCV build does not provide SIFT. Install a current opencv-python "
            "or opencv-python-headless package, then restart the kernel."
        )
    if params.feature_descriptor_size != 4 or params.feature_descriptor_orientation_bins != 8:
        raise ValueError(
            "OpenCV SIFT uses the standard 4x4 descriptor grid with 8 orientation bins; "
            "set feature_descriptor_size=4 and feature_descriptor_orientation_bins=8."
        )
    return cv2.SIFT_create(
        nfeatures=0,
        nOctaveLayers=params.steps_per_scale_octave,
        contrastThreshold=params.contrast_threshold,
        edgeThreshold=params.edge_threshold,
        sigma=params.initial_gaussian_blur,
    )


def extract_features(image: np.ndarray, sift: cv2.SIFT, params: SIFTParameters) -> FeatureSet:
    prepared = prepare_registration_image(image, params.registration_percentiles)
    height, width = prepared.shape
    scale = min(1.0, params.maximum_image_size / max(height, width))
    if scale < 1.0:
        prepared = cv2.resize(prepared, dsize=None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
    keypoints, descriptors = sift.detectAndCompute(prepared, None)
    if not keypoints or descriptors is None:
        return FeatureSet(np.empty((0, 2), np.float64), np.empty(0, np.float64), None, 0)
    xy = np.asarray([kp.pt for kp in keypoints], dtype=np.float64) / scale
    sizes = np.asarray([kp.size for kp in keypoints], dtype=np.float64) / scale
    return FeatureSet(xy, sizes, np.asarray(descriptors, dtype=np.float32), len(keypoints))


def fit_rigid_transform(source_xy: np.ndarray, target_xy: np.ndarray) -> np.ndarray:
    """Least-squares rigid transform mapping source points to target points."""
    source_xy = np.asarray(source_xy, dtype=np.float64)
    target_xy = np.asarray(target_xy, dtype=np.float64)
    if source_xy.shape != target_xy.shape or source_xy.ndim != 2 or source_xy.shape[1] != 2:
        raise ValueError("source_xy and target_xy must both have shape (n, 2).")
    if len(source_xy) < 2:
        raise ValueError("At least two point pairs are needed for a 2-D rigid transform.")
    source_center = source_xy.mean(axis=0)
    target_center = target_xy.mean(axis=0)
    covariance = (source_xy - source_center).T @ (target_xy - target_center)
    u, _, vt = np.linalg.svd(covariance)
    rotation = vt.T @ u.T
    if np.linalg.det(rotation) < 0:
        vt[-1, :] *= -1
        rotation = vt.T @ u.T
    translation = target_center - rotation @ source_center
    matrix = np.eye(3, dtype=np.float64)
    matrix[:2, :2] = rotation
    matrix[:2, 2] = translation
    return matrix


def transform_points(points_xy: np.ndarray, matrix: np.ndarray) -> np.ndarray:
    points_h = np.column_stack((np.asarray(points_xy, dtype=np.float64), np.ones(len(points_xy))))
    return (points_h @ np.asarray(matrix, dtype=np.float64).T)[:, :2]


def rigid_ransac(
    source_xy: np.ndarray,
    target_xy: np.ndarray,
    *,
    residual_threshold: float,
    min_inlier_ratio: float,
    max_trials: int,
    seed: int,
) -> tuple[np.ndarray | None, np.ndarray, np.ndarray]:
    source_xy = np.asarray(source_xy, dtype=np.float64)
    target_xy = np.asarray(target_xy, dtype=np.float64)
    n = len(source_xy)
    empty_mask = np.zeros(n, dtype=bool)
    empty_residuals = np.full(n, np.inf, dtype=np.float64)
    if n < 2:
        return None, empty_mask, empty_residuals

    required = max(2, int(math.ceil(min_inlier_ratio * n)))
    rng = np.random.default_rng(seed)
    best_matrix: np.ndarray | None = None
    best_mask = empty_mask
    best_residuals = empty_residuals
    best_score = (-1, -np.inf)

    for _ in range(max_trials):
        sample = rng.choice(n, size=2, replace=False)
        if np.linalg.norm(source_xy[sample[0]] - source_xy[sample[1]]) < 1e-6:
            continue
        if np.linalg.norm(target_xy[sample[0]] - target_xy[sample[1]]) < 1e-6:
            continue
        try:
            matrix = fit_rigid_transform(source_xy[sample], target_xy[sample])
        except (ValueError, np.linalg.LinAlgError):
            continue
        residuals = np.linalg.norm(transform_points(source_xy, matrix) - target_xy, axis=1)
        mask = residuals <= residual_threshold
        count = int(mask.sum())
        if count < 2:
            continue
        median = float(np.median(residuals[mask]))
        score = (count, -median)
        if score > best_score:
            best_score = score
            best_matrix = matrix
            best_mask = mask
            best_residuals = residuals
            if count == n:
                break

    if best_matrix is None or int(best_mask.sum()) < required:
        return None, best_mask, best_residuals

    # Local optimization: refit on all inliers until membership stabilizes.
    mask = best_mask
    matrix = best_matrix
    for _ in range(10):
        try:
            matrix = fit_rigid_transform(source_xy[mask], target_xy[mask])
        except (ValueError, np.linalg.LinAlgError):
            break
        residuals = np.linalg.norm(transform_points(source_xy, matrix) - target_xy, axis=1)
        new_mask = residuals <= residual_threshold
        if int(new_mask.sum()) < required:
            break
        if np.array_equal(new_mask, mask):
            mask = new_mask
            break
        mask = new_mask

    matrix = fit_rigid_transform(source_xy[mask], target_xy[mask])
    residuals = np.linalg.norm(transform_points(source_xy, matrix) - target_xy, axis=1)
    mask = residuals <= residual_threshold
    if int(mask.sum()) < required:
        return None, mask, residuals
    return matrix, mask, residuals


def match_and_estimate_rigid(
    moving: FeatureSet,
    fixed: FeatureSet,
    params: SIFTParameters,
    *,
    seed: int,
) -> tuple[np.ndarray | None, dict[str, Any]]:
    info: dict[str, Any] = {
        "moving_features": moving.n_features,
        "fixed_features": fixed.n_features,
        "candidate_matches": 0,
        "inliers": 0,
        "inlier_ratio": 0.0,
        "median_inlier_error_px": np.nan,
        "max_inlier_error_px": np.nan,
        "status": "failed",
    }
    if moving.descriptors is None or fixed.descriptors is None:
        info["status"] = "no SIFT descriptors"
        return None, info
    if len(moving.descriptors) < 2 or len(fixed.descriptors) < 2:
        info["status"] = "too few SIFT descriptors"
        return None, info

    matcher = cv2.BFMatcher(cv2.NORM_L2, crossCheck=False)
    knn_matches = matcher.knnMatch(moving.descriptors, fixed.descriptors, k=2)
    accepted: list[cv2.DMatch] = []
    for pair in knn_matches:
        if len(pair) < 2:
            continue
        closest, next_closest = pair
        if closest.distance >= params.closest_next_closest_ratio * next_closest.distance:
            continue
        moving_size = moving.sizes[closest.queryIdx]
        fixed_size = fixed.sizes[closest.trainIdx]
        if moving_size <= 0 or fixed_size <= 0:
            continue
        scale_ratio = max(moving_size / fixed_size, fixed_size / moving_size)
        if scale_ratio <= params.max_feature_scale_ratio:
            accepted.append(closest)

    info["candidate_matches"] = len(accepted)
    if len(accepted) < 2:
        info["status"] = "too few ratio-test matches"
        return None, info

    source_xy = np.asarray([moving.keypoints_xy[m.queryIdx] for m in accepted], dtype=np.float64)
    target_xy = np.asarray([fixed.keypoints_xy[m.trainIdx] for m in accepted], dtype=np.float64)
    matrix, inlier_mask, residuals = rigid_ransac(
        source_xy,
        target_xy,
        residual_threshold=params.maximal_alignment_error,
        min_inlier_ratio=params.minimum_inlier_ratio,
        max_trials=params.max_trials,
        seed=seed,
    )
    n_inliers = int(inlier_mask.sum())
    info["inliers"] = n_inliers
    info["inlier_ratio"] = n_inliers / len(accepted) if accepted else 0.0
    if n_inliers:
        info["median_inlier_error_px"] = float(np.median(residuals[inlier_mask]))
        info["max_inlier_error_px"] = float(np.max(residuals[inlier_mask]))
    if matrix is None:
        info["status"] = "RANSAC did not find an accepted rigid model"
        return None, info
    info["status"] = "ok"
    return matrix, info


def warp_image(image: np.ndarray, source_to_destination: np.ndarray, output_shape: tuple[int, int]) -> np.ndarray:
    height, width = output_shape
    return cv2.warpAffine(
        np.asarray(image, dtype=np.float32),
        np.asarray(source_to_destination[:2], dtype=np.float64),
        dsize=(width, height),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0,
    )


def warp_channels(channels_yx: np.ndarray, matrix: np.ndarray, output_shape: tuple[int, int]) -> np.ndarray:
    return np.stack([warp_image(channel, matrix, output_shape) for channel in channels_yx], axis=0)


def normalized_cross_correlation(a: np.ndarray, b: np.ndarray) -> float:
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    mask = np.isfinite(a) & np.isfinite(b) & ((a != 0) | (b != 0))
    if int(mask.sum()) < 16:
        return float("nan")
    x = a[mask] - a[mask].mean()
    y = b[mask] - b[mask].mean()
    denom = np.linalg.norm(x) * np.linalg.norm(y)
    return float((x @ y) / denom) if denom > 0 else float("nan")


def matrix_metrics(matrix: np.ndarray, prefix: str) -> dict[str, float]:
    return {
        f"{prefix}_rotation_deg": float(np.degrees(np.arctan2(matrix[1, 0], matrix[0, 0]))),
        f"{prefix}_dx_px": float(matrix[0, 2]),
        f"{prefix}_dy_px": float(matrix[1, 2]),
    }


def align_multichannel_sequence(
    sequence_ncyx: np.ndarray,
    registration_channel: int,
    params: SIFTParameters,
    *,
    sequence_name: str,
    failure_policy: str = "reuse_previous",
) -> tuple[np.ndarray, pd.DataFrame, list[np.ndarray]]:
    """Consecutively align frames, matching frame i to frame i-1.

    The pair transform maps frame i to frame i-1. Cumulative transforms map each
    original frame into frame 0 coordinates and are applied identically to all channels.
    This mirrors Fiji's Linear Stack Alignment with SIFT MultiChannel behavior.
    """
    sequence = np.asarray(sequence_ncyx)
    if sequence.ndim != 4:
        raise ValueError(f"Expected sequence shape (N,C,Y,X), got {sequence.shape}.")
    n_frames, n_channels, height, width = sequence.shape
    if not (0 <= registration_channel < n_channels):
        raise IndexError(f"registration_channel={registration_channel} outside 0..{n_channels-1}.")
    if failure_policy not in {"reuse_previous", "raise"}:
        raise ValueError("failure_policy must be 'reuse_previous' or 'raise'.")

    sift = build_sift(params)
    features = [extract_features(sequence[i, registration_channel], sift, params) for i in range(n_frames)]
    aligned = np.empty(sequence.shape, dtype=np.float32)
    aligned[0] = sequence[0].astype(np.float32, copy=False)
    cumulative = np.eye(3, dtype=np.float64)
    cumulative_matrices = [cumulative.copy()]
    rows: list[dict[str, Any]] = []

    first_row: dict[str, Any] = {
        "sequence": sequence_name,
        "frame": 0,
        "fixed_frame": 0,
        "moving_features": features[0].n_features,
        "fixed_features": features[0].n_features,
        "candidate_matches": np.nan,
        "inliers": np.nan,
        "inlier_ratio": np.nan,
        "median_inlier_error_px": np.nan,
        "max_inlier_error_px": np.nan,
        "ncc_before": 1.0,
        "ncc_after_pair_transform": 1.0,
        "status": "reference frame",
    }
    first_row.update(matrix_metrics(np.eye(3), "pair"))
    first_row.update(matrix_metrics(cumulative, "cumulative"))
    rows.append(first_row)

    for frame in range(1, n_frames):
        pair_matrix, info = match_and_estimate_rigid(
            features[frame],
            features[frame - 1],
            params,
            seed=params.random_seed + frame,
        )
        if pair_matrix is None:
            message = f"{sequence_name}: registration failed for frame {frame} -> {frame-1}: {info['status']}"
            if failure_policy == "raise":
                raise RuntimeError(message)
            warnings.warn(message + "; reusing the previous cumulative transform.", RuntimeWarning)
            pair_matrix = np.eye(3, dtype=np.float64)
            info["status"] = info["status"] + "; identity increment used"

        # mpicbg RigidModel2D.concatenate semantics: cumulative <- cumulative @ pair.
        cumulative = cumulative @ pair_matrix
        cumulative_matrices.append(cumulative.copy())
        aligned[frame] = warp_channels(sequence[frame], cumulative, (height, width))

        pair_warped = warp_image(sequence[frame, registration_channel], pair_matrix, (height, width))
        row: dict[str, Any] = {
            "sequence": sequence_name,
            "frame": frame,
            "fixed_frame": frame - 1,
            **info,
            "ncc_before": normalized_cross_correlation(
                sequence[frame, registration_channel], sequence[frame - 1, registration_channel]
            ),
            "ncc_after_pair_transform": normalized_cross_correlation(
                pair_warped, sequence[frame - 1, registration_channel]
            ),
        }
        row.update(matrix_metrics(pair_matrix, "pair"))
        row.update(matrix_metrics(cumulative, "cumulative"))
        rows.append(row)

    return aligned, pd.DataFrame(rows), cumulative_matrices


## Output and workflow functions

All output images are written as 32-bit floating-point TIFFs. This avoids integer truncation after interpolation and intensity normalization. ImageJ hyperstacks are written with explicit `ZCYX` metadata.

In [ ]:
def normalize_section_intensity(mass_stacks_csyx: np.ndarray) -> tuple[np.ndarray, pd.DataFrame]:
    data = np.asarray(mass_stacks_csyx, dtype=np.float32)
    if data.ndim != 4:
        raise ValueError(f"Expected mass stacks shape (C,S,Y,X), got {data.shape}.")
    means = data.mean(axis=(2, 3), dtype=np.float64)
    global_means = means.mean(axis=1)
    factors = np.ones_like(means, dtype=np.float64)
    valid = means > 0
    factors[valid] = np.broadcast_to(global_means[:, None], means.shape)[valid] / means[valid]
    normalized = data * factors[:, :, None, None]
    rows = []
    for channel in range(data.shape[0]):
        for section in range(data.shape[1]):
            rows.append(
                {
                    "channel": channel,
                    "section_index": section,
                    "section_mean_before": means[channel, section],
                    "target_mean": global_means[channel],
                    "multiplication_factor": factors[channel, section],
                    "section_mean_after": float(normalized[channel, section].mean(dtype=np.float64)),
                }
            )
    return normalized.astype(np.float32, copy=False), pd.DataFrame(rows)


def write_imagej_tiff(path: str | Path, data: np.ndarray, axes: str) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tifffile.imwrite(
        path,
        np.asarray(data, dtype=np.float32),
        imagej=True,
        photometric="minisblack",
        metadata={"axes": axes, "unit": "pixel"},
    )
    return path


def save_transform_matrices(path: str | Path, matrices: Sequence[np.ndarray], labels: Sequence[str]) -> Path:
    payload = {
        label: np.asarray(matrix, dtype=float).tolist()
        for label, matrix in zip(labels, matrices, strict=True)
    }
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    return path


def plot_registration_trace(qc: pd.DataFrame, path: str | Path, title: str) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig, axes = plt.subplots(3, 1, figsize=(8, 8), sharex=True)
    x = qc["frame"]
    axes[0].plot(x, qc["cumulative_dx_px"], marker="o")
    axes[0].set_ylabel("cumulative dx (px)")
    axes[1].plot(x, qc["cumulative_dy_px"], marker="o")
    axes[1].set_ylabel("cumulative dy (px)")
    axes[2].plot(x, qc["cumulative_rotation_deg"], marker="o")
    axes[2].set_ylabel("rotation (deg)")
    axes[2].set_xlabel("frame")
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)
    return path


def plot_reference_montage(stack_syx: np.ndarray, section_labels: Sequence[str], path: str | Path, title: str) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    n = len(stack_syx)
    cols = min(4, n)
    rows = int(math.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(3.2 * cols, 3.2 * rows), squeeze=False)
    finite = np.asarray(stack_syx)[np.isfinite(stack_syx)]
    vmin, vmax = np.percentile(finite, (1, 99.5)) if finite.size else (0, 1)
    for i, ax in enumerate(axes.flat):
        if i < n:
            ax.imshow(stack_syx[i], cmap="gray", vmin=vmin, vmax=vmax)
            ax.set_title(section_labels[i])
        ax.axis("off")
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)
    return path


def run_pipeline(
    input_dir: str | Path = "input_nrrd",
    output_root: str | Path = "output_python_pipeline",
    *,
    registration_mass: str = "12C14N",
    normalize_intensity: bool = True,
    save_aligned_plane_stacks: bool = False,
    failure_policy: str = "reuse_previous",
    params: SIFTParameters | None = None,
    verbose: bool = True,
) -> dict[str, Any]:
    params = params or SIFTParameters()
    input_dir = Path(input_dir).expanduser().resolve()
    output_root = Path(output_root).expanduser().resolve()

    def log(message: str) -> None:
        if verbose:
            print(message, flush=True)
    section_sum_dir = output_root / "01_aligned_plane_sums"
    aligned_plane_dir = output_root / "01b_aligned_plane_stacks"
    mass_stack_dir = output_root / "02_mass_sum_stacks"
    normalized_dir = output_root / "03_intensity_normalized_mass_stacks"
    hyperstack_dir = output_root / "04_hyperstacks"
    registered_mass_dir = output_root / "05_registered_mass_stacks"
    qc_dir = output_root / "qc"
    for directory in (
        section_sum_dir,
        mass_stack_dir,
        normalized_dir,
        hyperstack_dir,
        registered_mass_dir,
        qc_dir,
    ):
        directory.mkdir(parents=True, exist_ok=True)
    if save_aligned_plane_stacks:
        aligned_plane_dir.mkdir(parents=True, exist_ok=True)

    nrrd_files = sorted(
        [path for path in input_dir.iterdir() if path.is_file() and path.suffix.casefold() in {".nrrd", ".nhdr"}],
        key=section_sort_key,
    )
    if not nrrd_files:
        raise FileNotFoundError(f"No .nrrd or .nhdr files were found in {input_dir}.")
    log(f"Found {len(nrrd_files)} NRRD file(s) in {input_dir}")

    section_sums: list[np.ndarray] = []
    section_labels: list[str] = []
    manifest_rows: list[dict[str, Any]] = []
    within_qc_frames: list[pd.DataFrame] = []
    within_transform_files: list[str] = []
    expected_masses: list[str] | None = None
    expected_yx: tuple[int, int] | None = None
    registration_index: int | None = None

    for section_index, path in enumerate(nrrd_files):
        log(f"[{section_index + 1}/{len(nrrd_files)}] Aligning acquisition planes: {path.name}")
        planes_zcyx, header, mass_names = load_openmims_nrrd(path)
        if expected_masses is None:
            expected_masses = mass_names
            if registration_mass not in mass_names:
                raise ValueError(f"Registration mass {registration_mass!r} is not in {mass_names}.")
            registration_index = mass_names.index(registration_mass)
            expected_yx = tuple(planes_zcyx.shape[-2:])
        else:
            if mass_names != expected_masses:
                raise ValueError(f"Mass order differs in {path.name}: {mass_names} vs {expected_masses}.")
            if tuple(planes_zcyx.shape[-2:]) != expected_yx:
                raise ValueError(f"Image size differs in {path.name}: {planes_zcyx.shape[-2:]} vs {expected_yx}.")
        assert registration_index is not None

        label = f"sect{section_number(path):02d}" if section_number(path) is not None else path.stem
        section_labels.append(label)
        aligned_zcyx, qc, matrices = align_multichannel_sequence(
            planes_zcyx,
            registration_index,
            params,
            sequence_name=path.name,
            failure_policy=failure_policy,
        )
        qc.insert(1, "section_index", section_index)
        qc.insert(2, "section_label", label)
        within_qc_frames.append(qc)
        matrix_path = qc_dir / f"within_{safe_name(path.stem)}_transforms.json"
        save_transform_matrices(matrix_path, matrices, [f"plane_{i:03d}" for i in range(len(matrices))])
        within_transform_files.append(str(matrix_path))

        if save_aligned_plane_stacks:
            write_imagej_tiff(
                aligned_plane_dir / f"Aligned_planes_{path.stem}.tif",
                aligned_zcyx,
                "ZCYX",
            )
        summed_cyx = np.sum(aligned_zcyx, axis=0, dtype=np.float64).astype(np.float32)
        section_sums.append(summed_cyx)
        write_imagej_tiff(
            section_sum_dir / f"SUM_Aligned_{path.name}.tif",
            summed_cyx,
            "CYX",
        )
        manifest_rows.append(
            {
                "section_index": section_index,
                "section_label": label,
                "section_number": section_number(path),
                "input_file": path.name,
                "planes": planes_zcyx.shape[0],
                "channels": planes_zcyx.shape[1],
                "height": planes_zcyx.shape[2],
                "width": planes_zcyx.shape[3],
                "max_tracking_delta_header": _header_lookup(header, "max_tracking_delta"),
            }
        )

    assert expected_masses is not None and registration_index is not None
    log("Building one serial-section stack per mass...")
    raw_csyx = np.stack(section_sums, axis=1)  # channel, section, y, x
    for channel, mass in enumerate(expected_masses):
        write_imagej_tiff(
            mass_stack_dir / f"{channel + 1}_{safe_name(mass)}_sum_stack.tif",
            raw_csyx[channel],
            "ZYX",
        )

    intensity_qc: pd.DataFrame | None = None
    if normalize_intensity:
        log("Normalizing each section to the mean intensity of its mass stack...")
        selected_csyx, intensity_qc = normalize_section_intensity(raw_csyx)
        intensity_qc["mass"] = intensity_qc["channel"].map(dict(enumerate(expected_masses)))
        intensity_qc["section_label"] = intensity_qc["section_index"].map(dict(enumerate(section_labels)))
        for channel, mass in enumerate(expected_masses):
            write_imagej_tiff(
                normalized_dir / f"{channel + 1}_{safe_name(mass)}_sum_stack_intnorm.tif",
                selected_csyx[channel],
                "ZYX",
            )
        intensity_qc.to_csv(qc_dir / "intensity_normalization_factors.csv", index=False)
    else:
        selected_csyx = raw_csyx

    unregistered_zcyx = np.transpose(selected_csyx, (1, 0, 2, 3))
    unregistered_path = write_imagej_tiff(
        hyperstack_dir / "unregistered_hyperstack.tif",
        unregistered_zcyx,
        "ZCYX",
    )

    log(f"Registering serial sections on mass {registration_mass!r}...")
    registered_zcyx, section_qc, section_matrices = align_multichannel_sequence(
        unregistered_zcyx,
        registration_index,
        params,
        sequence_name="serial_sections",
        failure_policy=failure_policy,
    )
    section_qc.insert(1, "section_label", section_labels)
    registered_path = write_imagej_tiff(
        hyperstack_dir / "registered_hyperstack.tif",
        registered_zcyx,
        "ZCYX",
    )
    save_transform_matrices(
        qc_dir / "serial_section_transforms.json",
        section_matrices,
        section_labels,
    )

    registered_csyx = np.transpose(registered_zcyx, (1, 0, 2, 3))
    registered_mass_paths: list[str] = []
    for channel, mass in enumerate(expected_masses):
        path = write_imagej_tiff(
            registered_mass_dir / f"{channel + 1}_{safe_name(mass)}_registered.tif",
            registered_csyx[channel],
            "ZYX",
        )
        registered_mass_paths.append(str(path))

    manifest = pd.DataFrame(manifest_rows)
    within_qc = pd.concat(within_qc_frames, ignore_index=True)
    manifest.to_csv(qc_dir / "section_manifest.csv", index=False)
    within_qc.to_csv(qc_dir / "within_nrrd_registration.csv", index=False)
    section_qc.to_csv(qc_dir / "serial_section_registration.csv", index=False)
    plot_registration_trace(section_qc, qc_dir / "serial_section_transform_trace.png", "Serial-section registration")
    plot_reference_montage(
        unregistered_zcyx[:, registration_index],
        section_labels,
        qc_dir / "unregistered_reference_channel_montage.png",
        f"Unregistered sections: {registration_mass}",
    )
    plot_reference_montage(
        registered_zcyx[:, registration_index],
        section_labels,
        qc_dir / "registered_reference_channel_montage.png",
        f"Registered sections: {registration_mass}",
    )

    run_manifest = {
        "input_dir": str(input_dir),
        "output_root": str(output_root),
        "input_files": [path.name for path in nrrd_files],
        "section_labels": section_labels,
        "mass_names": expected_masses,
        "registration_mass": registration_mass,
        "registration_channel_zero_based": registration_index,
        "normalize_intensity": normalize_intensity,
        "save_aligned_plane_stacks": save_aligned_plane_stacks,
        "failure_policy": failure_policy,
        "sift_parameters": asdict(params),
        "array_conventions": {
            "loaded_nrrd": "ZCYX (acquisition plane, mass/channel, y, x)",
            "mass_stacks": "CZYX (mass/channel, serial section, y, x)",
            "imagej_hyperstack": "ZCYX (serial section, mass/channel, y, x)",
        },
        "outputs": {
            "unregistered_hyperstack": str(unregistered_path),
            "registered_hyperstack": str(registered_path),
            "registered_mass_stacks": registered_mass_paths,
            "within_nrrd_transform_files": within_transform_files,
        },
    }
    (output_root / "run_manifest.json").write_text(json.dumps(run_manifest, indent=2), encoding="utf-8")
    log(f"Done. Outputs written to {output_root}")

    return {
        "mass_names": expected_masses,
        "section_labels": section_labels,
        "raw_mass_stacks_csyx": raw_csyx,
        "selected_mass_stacks_csyx": selected_csyx,
        "registered_hyperstack_zcyx": registered_zcyx,
        "section_manifest": manifest,
        "within_nrrd_qc": within_qc,
        "serial_section_qc": section_qc,
        "intensity_qc": intensity_qc,
        "output_root": output_root,
    }


## Configuration

Place the NRRD files in a folder named `input_nrrd` beside this notebook. Normally, only this cell needs editing.

Environment variables `NANOSIMS_INPUT_DIR` and `NANOSIMS_OUTPUT_ROOT` can override the two folder paths, which is useful for automated or remote execution.

In [ ]:
INPUT_DIR = Path(os.environ.get("NANOSIMS_INPUT_DIR", "input_nrrd"))
OUTPUT_ROOT = Path(os.environ.get("NANOSIMS_OUTPUT_ROOT", "output_python_pipeline"))

# Use the mass name from the NRRD header rather than a fragile hard-coded channel index.
REGISTRATION_MASS = "12C14N"

# This is Step 3 from the supplied notebook. Set False to register unnormalized sums.
NORMALIZE_INTENSITY = True

# Plane-aligned 4-D files can be large. The summed per-section files are always saved.
SAVE_ALIGNED_PLANE_STACKS = False

# "reuse_previous" matches FIJI's practical behavior if a pair has no accepted model:
# use an identity increment, retain the previous cumulative transform, and emit a warning.
# Set to "raise" to stop immediately instead.
FAILURE_POLICY = "reuse_previous"

SIFT_PARAMS = SIFTParameters(
    initial_gaussian_blur=1.60,
    steps_per_scale_octave=3,
    minimum_image_size=64,
    maximum_image_size=1024,
    feature_descriptor_size=4,
    feature_descriptor_orientation_bins=8,
    closest_next_closest_ratio=0.92,
    maximal_alignment_error=25.0,
    minimum_inlier_ratio=0.05,
    max_trials=1000,
    max_feature_scale_ratio=1.5,
    # Python/OpenCV-specific controls:
    contrast_threshold=0.04,
    edge_threshold=10.0,
    registration_percentiles=(0.5, 99.5),
    random_seed=1729,
)

print(f"Input folder : {INPUT_DIR.resolve()}")
print(f"Output folder: {OUTPUT_ROOT.resolve()}")

## Run the complete pipeline

Runtime depends mainly on the number and size of acquisition planes. Progress is printed once per NRRD.

In [ ]:
results = run_pipeline(
    input_dir=INPUT_DIR,
    output_root=OUTPUT_ROOT,
    registration_mass=REGISTRATION_MASS,
    normalize_intensity=NORMALIZE_INTENSITY,
    save_aligned_plane_stacks=SAVE_ALIGNED_PLANE_STACKS,
    failure_policy=FAILURE_POLICY,
    params=SIFT_PARAMS,
    verbose=True,
)

print("\nCompleted")
print("Masses  :", results["mass_names"])
print("Sections:", results["section_labels"])
print("Registered hyperstack shape (Z, C, Y, X):", results["registered_hyperstack_zcyx"].shape)

## Review registration QC

The first table confirms file order, plane counts, and dimensions. The second summarizes all within-NRRD plane registrations. The third contains pairwise and cumulative transforms for serial-section registration.

`ncc_after_pair_transform` is a diagnostic, not an acceptance criterion: adjacent biological sections can differ enough that correlation is imperfect even when landmarks agree.

In [ ]:
from IPython.display import Image, display

display(results["section_manifest"])

within = results["within_nrrd_qc"].copy()
within_summary = (
    within.groupby("section_label", sort=False)
    .agg(
        planes=("frame", "count"),
        accepted_models=("status", lambda s: int((s == "ok").sum())),
        minimum_candidate_matches=("candidate_matches", "min"),
        minimum_inliers=("inliers", "min"),
        median_ncc_before=("ncc_before", "median"),
        median_ncc_after=("ncc_after_pair_transform", "median"),
        maximum_abs_cumulative_dx_px=("cumulative_dx_px", lambda s: float(s.abs().max())),
        maximum_abs_cumulative_dy_px=("cumulative_dy_px", lambda s: float(s.abs().max())),
        maximum_abs_cumulative_rotation_deg=("cumulative_rotation_deg", lambda s: float(s.abs().max())),
    )
    .reset_index()
)
display(within_summary)

display(
    results["serial_section_qc"][[
        "section_label",
        "frame",
        "fixed_frame",
        "candidate_matches",
        "inliers",
        "inlier_ratio",
        "median_inlier_error_px",
        "ncc_before",
        "ncc_after_pair_transform",
        "pair_rotation_deg",
        "pair_dx_px",
        "pair_dy_px",
        "cumulative_rotation_deg",
        "cumulative_dx_px",
        "cumulative_dy_px",
        "status",
    ]]
)

failed = within.loc[~within["status"].isin(["reference frame", "ok"])]
serial_failed = results["serial_section_qc"].loc[
    ~results["serial_section_qc"]["status"].isin(["reference frame", "ok"])
]
if failed.empty and serial_failed.empty:
    print("All non-reference registration pairs produced accepted rigid models.")
else:
    print("Review registration warnings in the detailed QC CSV files.")

qc_dir = Path(results["output_root"]) / "qc"
for image_name in (
    "unregistered_reference_channel_montage.png",
    "registered_reference_channel_montage.png",
    "serial_section_transform_trace.png",
):
    image_path = qc_dir / image_name
    if image_path.exists():
        display(Image(filename=str(image_path)))

## Output layout

The pipeline creates:

```text
output_python_pipeline/
├── 01_aligned_plane_sums/               # one 7-channel summed TIFF per NRRD
├── 01b_aligned_plane_stacks/             # optional; only when enabled
├── 02_mass_sum_stacks/                   # raw serial-section stack per mass
├── 03_intensity_normalized_mass_stacks/  # optional normalized stack per mass
├── 04_hyperstacks/
│   ├── unregistered_hyperstack.tif
│   └── registered_hyperstack.tif
├── 05_registered_mass_stacks/            # final registered stack per mass
├── qc/                                   # CSV, JSON transforms, and PNG diagnostics
└── run_manifest.json                     # parameters, order, axes, and output paths
```

Array conventions used internally:

- loaded NRRD: `ZCYX` = acquisition plane, mass, Y, X
- mass stacks: `CZYX` = mass, serial section, Y, X
- ImageJ hyperstacks: `ZCYX` = serial section, mass, Y, X

The final analysis-ready products are in `05_registered_mass_stacks/`; the full multichannel result is `04_hyperstacks/registered_hyperstack.tif`.